# Ps3_NeuroGeo Pipeline

This notebook executes the full NeuroGeo-Field pipeline (Phases 0-5), integrating the candidate generation, validation, reranking, and navigation components.

### Phase 0: Baseline Reproduction

In [1]:
!python scripts/run_phase0.py

NEUROGEO-FIELD :: PHASE 0 VERIFICATION & BASELINE REPRODUCTION

[Step 1/4] Validating source data contracts and table integrity...
  + addresses.csv             : OK (398.7 KB)
  + baseline_geocodes.csv     : OK (92.5 KB)
  + field_visits.csv          : OK (943.6 KB)
  + visit_gps_points.csv      : OK (7125.7 KB)
  + landmarks_poi.csv         : OK (11.2 KB)
  + localities.csv            : OK (1.7 KB)
  + towns.csv                 : OK (0.1 KB)
  + surveyed_addresses.csv    : OK (2.4 KB)
  + splits.csv                : OK (38.9 KB)
  => All primary keys, row counts, and schema constraints verified.

[Step 2/4] Validating official 100-address benchmark partition...
  + Benchmark Count  : 100 addresses
  + Train Split      : 66 (expected 66)
  + Validation Split : 19 (expected 19)
  + Test Split       : 15 (expected 15)
  + Visited Subgroup : 49 (expected 49)
  + Cold-Start Subgrp: 51 (expected 51)
  => Zero overlap verified across train, validation, and test splits.

[Step 3/4] Testing z

### Phase 1: Visit Sanitizer & Miner

In [1]:
!python scripts/run_phase1.py

NEUROGEO-FIELD :: PHASE 1 OPERATIONAL SANITIZER & TRAJECTORY MINER

[Step 1/5] Loading field visits and GPS trajectory breadcrumbs...
  + Field Visits Count : 5578 visits across 1477 unique addresses
  + GPS Breadcrumbs    : 160406 points across 5578 unique visits
  + Addresses Ingested : 3117 master records

[Step 2/5] Mining GPS trajectories, tortuosity, and dwell medoids...
  => Extracted trajectory features for 5578 visits in 23.69s.

[Step 3/5] Sanitizing operational visits & computing I(v), S(v)...
  + High Integrity Visits (I >= 0.70) : 4897 (87.8%)
  + Compromised Visits   (I < 0.10)  : 162 (2.9%)
  + Credible Doorstep Observations    : 3400
  + Shifted-Borrower Observations     : 455 (address-state evidence)
  + Untraceable Search Loops          : 1400 (negative search evidence)

[Step 4/5] Computing address-level doorstep medoids...
  + Total Visited Addresses with Medoids : 1477
  + Addresses with Credible Doorsteps    : 1153 (78.1%)

[Step 5/5] Evaluating Model 1 Milestone 

### Phase 2: Address Parser & POI

In [1]:
!python scripts/run_phase2.py

NeuroGeo-Field: Phase 2 Execution
Multilingual Script-Aware Address Parser & Locality-Bounded POI Resolver
Repository Root: Ps3_NeuroGeo
Outputs Directory: reports\metrics

[Step 1/5] Ingesting address and geospatial reference datasets...
  * Addresses loaded: 3,117 rows
  * Localities loaded: 36 rows (36 administrative neighborhoods)
  * Landmarks (POIs) loaded: 240 rows
  * Towns loaded: 3 rows (T1, T2, T3)

[Step 2/5] Verifying Phase 2 structural invariants...
  * Max duplicate POI types per locality disc: 1 (Required: <= 1)
  [PASS] Zero-Duplicate Locality Invariant verified (0 duplicate POIs per type per locality).

[Step 3/5] Parsing addresses with multilingual normalizer and POI resolver...
  * Parsed 3,117 addresses in 4.19s (743.7 addr/s)

[Step 4/5] Computing entity resolution diagnostics & quality scoreboard...
----------------------------------------------------------------------
PHASE 2 ENTITY RESOLUTION SCOREBOARD:
---------------------------------------------------------

### Phase 3: Spatial Graph & Validation

In [1]:
!python scripts/run_phase3.py

NeuroGeo-Field: Phase 3 Execution
Hierarchical Spatial Graph & Leakage-Safe Cold-Start Validation (SCV & LOSO)
Repository Root: Ps3_NeuroGeo
Outputs Directory: reports\metrics

[Step 1/5] Ingesting Phase 1, Phase 2, and reference datasets...
  * Address entities: 3,117 rows
  * Visited doorstep medoids: 1,477 rows (1,477 visited accounts)
  * Visit features: 5,578 rows
  * Localities: 36 neighborhoods
  * Commercial baseline priors: 2,880 rows
  * Surveyed benchmark addresses: 100 ground truth rows

[Step 2/5] Verifying Phase 3 structural invariants & provenance tags...
  [PASS] Dataset row counts and fit_scope provenance schema verified.

[Step 3/5] Constructing hierarchical spatial graph (Town -> Locality -> Street)...
  * Graph constructed in 0.36s
  * Total graph nodes: 687
  * Total graph edges: 684
  [PASS] Structural hierarchy invariant verified: 100% of nodes start with 'T'.
  [PASS] Zero cross-locality hub collisions verified.
  * Exported street centroids (fit_scope='producti

### Phase 4: Calibration & Reranking

In [1]:
!python scripts/run_phase4.py

NeuroGeo-Field: Phase 4 Execution
Candidate Generation/Reranking & Leakage-Safe Empirical Quantile Calibration
Repository Root: Ps3_NeuroGeo
Outputs Directory: reports\metrics

[Step 1/6] Ingesting Phase 1-3 artifacts and reference data...
  * Portfolio addresses: 3,117 rows
  * Visited doorstep medoids: 1,477 rows
  * Visit features: 5,578 rows
  * Commercial baseline priors: 2,880 rows
  * Localities: 36 neighborhoods
  * POI Landmarks: 240 reference features
  * Surveyed benchmark addresses: 100 ground truth rows
  * SCV visited-address episodes: 1,477 episodes (1,477 unique addresses across 1,339 unique visited accounts)

[Step 2/6] Building hierarchical spatial graph for candidate C4 extraction...
  * Graph constructed: 687 nodes, 684 edges

[Step 3/6] Fitting deterministic linear reranker weights on Train split (N=66)...
  * Benchmark Split counts: Train=66, Val=19, Test=15
  * Reranker weights fitted on Train (N=66):
      w_text: 5.0000
      w_visit: 4.2412
      w_neigh: 1.50

### Phase 5: Navigator & Evidence Cards

In [1]:
!python scripts/run_phase5.py

NeuroGeo-Field: Phase 5 Execution
Offline Deterministic Navigator & Auditable Evidence Cards
Repository Root: Ps3_NeuroGeo
Outputs Directory: reports\metrics

[Step 1/4] Ingesting frozen Phase 4 predictions and local evidence artifacts...
  * Phase 4 predictions ingested: 100 rows (authoritative)
  * Address entity metadata: 3117 addresses
  * Operational visit records: 5578 visits across 1477 addresses

[Step 2/4] Generating deterministic auditable evidence cards...
  * Exported: reports\metrics\phase5_evidence_cards.json (100 cards)
  * Status distribution: {'LOCALITY_BOUNDED': 15, 'DOORSTEP_VERIFIED': 45, 'STREET_RESOLVED': 34, 'BASELINE_FALLBACK': 6}
  * Cards with negative field warnings: 25

[Step 3/4] Generating deterministic navigation scenarios across synthetic agent positions...
  * Exported: reports\metrics\phase5_navigation_examples.csv (400 instructions)
  * Navigation mode distribution: {'LOCALITY': 15, 'DOORSTEP': 45, 'STREET': 34, 'BASELINE': 6}

[Step 4/4] Verifying Ph